# RoadScan AI — free pothole training pipeline

This notebook builds a pothole-only YOLOv8 model from public road-damage datasets and exports ONNX for RoadScan AI. It is designed for a free Google Colab GPU.

Datasets: RDD2022 (47,420 images; pothole class D40) plus the 1,243-image IVCNZ pothole dataset. RDD2022 covers Japan, India, Czech Republic, Norway, US and China. The training pipeline also enables blur, noise, brightness and mosaic augmentation to improve robustness to phone vibration, motion blur and low-light footage.


In [ ]:
!pip -q install ultralytics==8.3.0 lxml
import os, zipfile, shutil, glob, random, xml.etree.ElementTree as ET
from pathlib import Path
ROOT=Path('/content/roadscan_train'); ROOT.mkdir(exist_ok=True)


In [ ]:
# Download the official RDD2022 archive.
import urllib.request
url='https://bigdatacup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022/RDD2022.zip'
rdd_zip=ROOT/'RDD2022.zip'
if not rdd_zip.exists(): urllib.request.urlretrieve(url, rdd_zip)
print('RDD2022:', rdd_zip.stat().st_size/1e9, 'GB')
with zipfile.ZipFile(rdd_zip) as z: z.extractall(ROOT/'rdd_raw')
print('RDD extracted')


In [ ]:
# Download the open 1,243-image YOLO pothole dataset.
url2='https://github.com/jaygala24/pothole-detection/releases/download/v1.0.0/Pothole.Dataset.IVCNZ.zip'
pzip=ROOT/'ivcnz.zip'
if not pzip.exists(): urllib.request.urlretrieve(url2,pzip)
with zipfile.ZipFile(pzip) as z: z.extractall(ROOT/'ivcnz')
print('IVCNZ extracted')


In [ ]:
# Convert RDD2022 Pascal-VOC D40 potholes to one-class YOLO labels and merge IVCNZ.
OUT=ROOT/'dataset';
for split in ['train','val','test']:
    (OUT/'images'/split).mkdir(parents=True,exist_ok=True); (OUT/'labels'/split).mkdir(parents=True,exist_ok=True)

def voc_to_yolo(xml_path, img_path):
    root=ET.parse(xml_path).getroot(); size=root.find('size')
    W=float(size.find('width').text); H=float(size.find('height').text); rows=[]
    for obj in root.findall('object'):
        name=obj.find('name').text
        if name!='D40': continue
        b=obj.find('bndbox'); x1=float(b.find('xmin').text); y1=float(b.find('ymin').text); x2=float(b.find('xmax').text); y2=float(b.find('ymax').text)
        xc=((x1+x2)/2)/W; yc=((y1+y2)/2)/H; bw=(x2-x1)/W; bh=(y2-y1)/H
        if bw>0 and bh>0: rows.append(f'0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')
    return rows

xmls=list((ROOT/'rdd_raw').rglob('*.xml')); random.seed(42); random.shuffle(xmls)
n=0
for xml in xmls:
    try: rows=voc_to_yolo(xml, None)
    except Exception: continue
    if not rows: continue
    stem=xml.stem
    candidates=list(xml.parent.glob(stem+'.jpg'))+list(xml.parent.glob(stem+'.JPG'))
    if not candidates: candidates=list((ROOT/'rdd_raw').rglob(stem+'.jpg'))
    if not candidates: continue
    img=candidates[0]; split='val' if n%10==0 else 'train'; dst=OUT/'images'/split/(f'rdd_{n}.jpg'); shutil.copy2(img,dst); (OUT/'labels'/split/f'rdd_{n}.txt').write_text('\n'.join(rows)); n+=1
print('RDD pothole images:',n)


In [ ]:
# Merge IVCNZ YOLO labels (class 0 = pothole).
imgs=list((ROOT/'ivcnz').rglob('*.jpg'))+list((ROOT/'ivcnz').rglob('*.JPG')); m=0
for img in imgs:
    lab=img.with_suffix('.txt')
    if not lab.exists(): continue
    split='val' if m%10==0 else 'train'; shutil.copy2(img,OUT/'images'/split/f'ivc_{m}.jpg'); shutil.copy2(lab,OUT/'labels'/split/f'ivc_{m}.txt'); m+=1
print('IVCNZ images:',m)


In [ ]:
(OUT/'data.yaml').write_text("path: %s\ntrain: images/train\nval: images/val\nnc: 1\nnames: ['pothole']\n" % OUT)
from ultralytics import YOLO
model=YOLO('yolov8n.pt')
results=model.train(data=str(OUT/'data.yaml'), epochs=80, imgsz=640, batch=16, device=0, workers=2, cache=False,
    hsv_h=0.02, hsv_s=0.75, hsv_v=0.55, degrees=4, translate=0.12, scale=0.45, shear=2,
    perspective=0.0008, fliplr=0.5, mosaic=1.0, mixup=0.10, copy_paste=0.10,
    patience=18, cos_lr=True, close_mosaic=10, project=str(ROOT/'runs'), name='roadscan_pothole')


In [ ]:
# Validate and export ONNX for the browser.
best=YOLO(str(ROOT/'runs/roadscan_pothole/weights/best.pt'))
metrics=best.val(data=str(OUT/'data.yaml'), imgsz=640, device=0)
print('mAP50:', metrics.box.map50, 'mAP50-95:', metrics.box.map)
exported=best.export(format='onnx', imgsz=640, opset=12, simplify=True, dynamic=False)
print('ONNX:', exported)


## Next step
Upload the generated `best.onnx` to the RoadScan AI model host and point `MODEL` in `app-v2.js` at it. Do not replace the production model until validation metrics and real phone/night test videos are checked.

RDD2022 is CC BY 4.0; the IVCNZ repository is MIT. Keep dataset/model attribution with the project. RDD2022's US subset has additional Google Street View copyright terms, so commercial redistribution should be reviewed before packaging those images or derived assets.
